In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
    collect_attributions,
    load_dyst_samples,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
split_name = "base40"
subsample_interval = 1

split_dir = os.path.join(DATA_DIR, split_name)
system_name = "Lorenz"
# ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

context_length = 512
context_start_time = 2000
context_end_time = context_start_time + context_length

# Prepare input series
sample_idx = 0
selected_dim = 0
dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(context.shape)

prediction_length = 512
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")


In [ ]:
pipeline.add_v_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])
pipeline.add_head_attribution_hooks([(i, j) for j in range(num_heads) for i in range(num_layers)])

preds = pipeline.predict_with_full_outputs(
    context, prediction_length=prediction_length, output_attentions=True, return_dict_in_generate=True
)

In [ ]:
def collect_ca_attention_scores(preds):
    dec_outs = preds[1]
    rollouts = len(dec_outs)

    num_layers = len(dec_outs[0].cross_attentions[0])
    num_samples, num_heads, _, context_len = dec_outs[0].cross_attentions[0][0].shape

    t = (rollouts - 1) * 64 + len(dec_outs[-1].cross_attentions)
    attention_scores = torch.zeros((t, num_layers, num_samples, num_heads, context_len))

    t_curr = 0
    for r in range(rollouts):
        for step in dec_outs[r].cross_attentions:
            for l in range(num_layers):
                attention_scores[t_curr, l] = step[l][:, :, -1, :]
            t_curr += 1

    return attention_scores


def collect_sa_attention_scores(preds):
    dec_outs = preds[1]
    rollouts = len(dec_outs)

    num_layers = len(dec_outs[0].cross_attentions[0])
    num_samples, num_heads, _, max_context_len = dec_outs[0].cross_attentions[-11][0].shape

    t = (rollouts - 1) * 64 + len(dec_outs[-1].cross_attentions)
    attention_scores = torch.empty((t, num_layers, num_samples, num_heads, max_context_len))

    t_curr = 0
    for r in range(rollouts):
        for step in dec_outs[r].cross_attentions:
            for l in range(num_layers):
                context_len_curr = step[l][:, :, -1, :].shape[-1]
                attention_scores[t_curr, l, :, :, :context_len_curr] = step[l][:, :, -1, :]
            t_curr += 1

    return attention_scores

In [ ]:
sample0 = collect_ca_attention_scores(preds)[:, :, 0, :, :]
sample0_sa = collect_sa_attention_scores(preds)[:, :, 0, :, :]

In [ ]:
# Plot the context series vs its index
context_np = context.squeeze(0).detach().cpu().numpy()
xs = np.arange(context_np.shape[-1])

plt.figure(figsize=(10, 3))
plt.plot(xs, context_np, linewidth=1.2)
plt.xlabel("context index")
plt.ylabel("value")
plt.title("Context series")
plt.tight_layout()

# Save and show
if "plot_save_dir" in globals():
    os.makedirs(plot_save_dir, exist_ok=True)
    plt.savefig(os.path.join(plot_save_dir, "context_series.png"), dpi=200)
plt.show()


In [ ]:
# Plot attention heatmap for layer 0 with 4x3 grid of heads: x=timestep, y=context index
assert sample0.ndim == 4, f"Expected 4D tensor [t, layers, heads, context], got {sample0.shape}"
T, L, H, C = sample0.shape
assert L == 12 and H == 12, f"Expected 12 layers and 12 heads, got {L} and {H}"

layer = 8

attn = sample0.detach().float().cpu().numpy()[:]  # [T, L, H, C]

fig, axes = plt.subplots(4, 3, figsize=(15, 18))
axes = axes.flatten()

for head in range(H):
    ax = axes[head]
    im = ax.imshow(attn[:, layer, head, :].T, aspect="auto", origin="lower", extent=[0, T, 0, C], cmap="viridis")
    ax.set_xlabel("timestep")
    ax.set_ylabel("context")
    ax.set_title(f"Layer {layer}, Head {head}")

    # Add colorbar for each subplot
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("attention score")

plt.tight_layout()

# Save and show
plt.savefig(os.path.join(plot_save_dir, f"attention_heatmap_L{layer}_all_heads.pdf"))
plt.show()


In [ ]:
np.save("../outputs/camaps.npy", attn)

In [ ]:
# Plot attention heatmap for a single head
layer = 3
head = 10

attn = sample0.detach().float().cpu().numpy()[:, layer, head, :]  # [T, C]
attnT = attn.T
log_attnT = np.log(attnT + 1e-10)  # Add small epsilon to avoid log(0)
vmin, vmax = np.min(log_attnT), np.max(log_attnT)
print(f"vmin: {vmin}, vmax: {vmax}")
plt.figure(figsize=(6, 6))
im = plt.imshow(log_attnT, aspect="equal", origin="lower", cmap="pink_r", vmin=vmin, vmax=vmax)
plt.xlabel("Prediction Timestep", fontweight="bold")
plt.ylabel("Context Timestep", fontweight="bold")
plt.title(f"Layer {layer}, Head {head} (log scale)", fontweight="bold")
plt.colorbar(im, label="Log Attention Score", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, f"attention_heatmap_L{layer}_H{head}_log.pdf"), bbox_inches="tight")

In [ ]:
# Plot attention heatmap for a single head
layer = 3
head = 10

attn = sample0.detach().float().cpu().numpy()[:, layer, head, :]  # [T, C]
attnT = attn.T
vmin, vmax = np.min(attnT), np.max(attnT)
print(f"vmin: {vmin}, vmax: {vmax}")
plt.figure(figsize=(6, 6))
im = plt.imshow(attnT, aspect="equal", origin="lower", cmap="bone_r", vmin=vmin, vmax=vmax)
plt.xlabel("Prediction Timestep", fontweight="bold")
plt.ylabel("Context Timestep", fontweight="bold")
plt.title(f"Layer {layer}, Head {head}", fontweight="bold")
plt.colorbar(im, label="Attention Score", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, f"attention_heatmap_L{layer}_H{head}.pdf"), bbox_inches="tight")

In [ ]:
head_outputs = {
    i: [collect_attributions(pipeline.ca_head_attribution_outputs[i][j]) for j in range(num_heads)]
    for i in range(num_layers)
}

In [ ]:
pipeline.ca_v_attribution_outputs[0][0][0].shape

In [ ]:
head_outputs[0][0].shape

In [ ]:
rollout = 0
sample = 0

for layer in range(3, 4):
    fig, axes = plt.subplots(3, 4, figsize=(24, 24))
    axes = axes.flatten()

    for head in range(num_heads):
        ax = axes[head]

        logits = (
            pipeline.unembed_residual(pipeline.ca_v_attribution_outputs[layer][head][rollout][sample])
            .detach()
            .cpu()
            .float()
            .numpy()
        )
        vabs = np.nanmax(np.abs(logits)) if np.any(np.isfinite(logits)) else 0

        im = ax.imshow(
            logits[:prediction_length, :].T,
            aspect="auto",
            cmap="RdBu",
            vmin=-vabs,
            vmax=vabs,
        )

        ax.invert_yaxis()
        ax.set_xlabel("Prediction Timestep")
        ax.set_ylabel("Vocab Index")
        ax.set_title(f"Layer {layer}, Head {head}")

        # Add colorbar for each subplot
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Logit Value")

    plt.suptitle(f"Cross-Attention Value Attribution (Layer {layer})", fontsize=16, y=1.00)
    plt.tight_layout()

    # Save and show
    if "plot_save_dir" in globals():
        os.makedirs(plot_save_dir, exist_ok=True)
        plt.savefig(os.path.join(plot_save_dir, f"ca_v_attribution_L{layer}_all_heads.png"), dpi=200)
    plt.close(fig)

In [ ]:
assert "sample0" in globals(), "sample0 not found. Run attention collection cells first."
attn = sample0.detach().float().cpu().numpy()  # [T, L, H, C]
T, L, H, C = attn.shape

# Compute entropy: H = -sum_c p log p
eps = 1e-12
p = attn / (attn.sum(axis=-1, keepdims=True) + eps)
ent_t_l_h = -(p * np.log(p + eps)).sum(axis=-1)  # [T, L, H]
avg_ent_l_h = ent_t_l_h.mean(axis=0)  # [L, H]

# Plot heatmap
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(-avg_ent_l_h, aspect="equal", cmap="viridis", origin="lower")
ax.set_xlabel("Head index", fontweight="bold")
ax.set_ylabel("Layer index", fontweight="bold")
ax.set_title("Avg Entropy per Head (CA)", fontweight="bold")
fig.colorbar(im, ax=ax, label="-Entropy (nats)", shrink=0.8)
plt.tight_layout()

plt.savefig(os.path.join(plot_save_dir, "ca_entropy_heatmap_LxH.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
# # Display attention heatmaps for top k and bottom k entropy heads
# assert "avg_ent_l_h" in globals() and "sample0" in globals(), "Run entropy and attention collection cells first."

# # Get entropy values and attention data
# neg_avg_entropy = -avg_ent_l_h  # shape [L, H]
# attn = sample0.detach().float().cpu().numpy()  # [T, L, H, C]
# T, L, H, C = attn.shape

# # Number of heads to display (3x3 grid)
# k = 9

# # Get top k (highest entropy) and bottom k (lowest entropy) heads
# entropy_flat = -neg_avg_entropy.reshape(-1)
# indices_flat = np.argsort(entropy_flat)
# bottom_k_indices = indices_flat[-k:]
# top_k_indices = indices_flat[:k]

# # Convert flat indices to (layer, head) tuples
# bottom_k_heads = [(idx // H, idx % H) for idx in bottom_k_indices]
# top_k_heads = [(idx // H, idx % H) for idx in top_k_indices]

# # Create figure with 3x3 grids side by side
# subplot_size = 3
# fig = plt.figure(figsize=(subplot_size * 6 + 1, subplot_size * 3 + 1))

# # Create GridSpec with space between the two grids
# import matplotlib.gridspec as gridspec

# gs = gridspec.GridSpec(
#     3,
#     6,
#     figure=fig,
#     wspace=0.3,
#     hspace=0.3,
#     width_ratios=[1, 1, 1, 1, 1, 1],
#     left=0.05,
#     right=0.98,
#     top=0.88,
#     bottom=0.08,
# )

# # Add titles for each grid
# fig.text(0.25, 0.92, "Lowest Entropy Heads", ha="center", fontsize=14, fontweight="bold")
# fig.text(0.75, 0.92, "Highest Entropy Heads", ha="center", fontsize=14, fontweight="bold")

# # Plot top k entropy heads (left 3x3 grid)
# for i, (l, h) in enumerate(top_k_heads):
#     row = i // 3
#     col = i % 3
#     ax = fig.add_subplot(gs[row, col])
#     im = ax.imshow(attn[:, l, h, :].T, aspect="equal", origin="lower", extent=(0, T, 0, C), cmap="bone_r")
#     ax.set_xlabel("Prediction Timestep", fontsize=9)
#     ax.set_xticks(np.arange(0, T + 128, 128))
#     ax.set_ylabel("Context Timestep", fontsize=9)
#     ax.set_yticks(np.arange(0, C, 128))
#     ax.set_title(f"Layer {l}, Head {h}\n(Entropy: {-neg_avg_entropy[l, h]:.2f})", fontsize=10)
#     ax.tick_params(labelsize=8)
#     cbar = fig.colorbar(im, ax=ax, shrink=0.8)
#     cbar.ax.tick_params(labelsize=7)

# # Plot bottom k entropy heads (right 3x3 grid)
# for i, (l, h) in enumerate(bottom_k_heads):
#     row = i // 3
#     col = i % 3 + 3  # Offset by 3 columns
#     ax = fig.add_subplot(gs[row, col])
#     im = ax.imshow(attn[:, l, h, :].T, aspect="equal", origin="lower", extent=(0, T, 0, C), cmap="bone_r")
#     ax.set_xlabel("Prediction Timestep", fontsize=9)
#     ax.set_xticks(np.arange(0, T + 128, 128))
#     ax.set_ylabel("Context Timestep", fontsize=9)
#     ax.set_yticks(np.arange(0, C, 128))
#     ax.set_title(f"Layer {l}, Head {h}\n(Entropy: {-neg_avg_entropy[l, h]:.2f})", fontsize=10)
#     ax.tick_params(labelsize=8)
#     cbar = fig.colorbar(im, ax=ax, shrink=0.8)
#     cbar.ax.tick_params(labelsize=7)

# plt.savefig(
#     os.path.join(plot_save_dir, f"attention_heatmap_top_bottom_{k}_entropy_heads.pdf"), bbox_inches="tight", dpi=200
# )
# plt.show()

# print(f"Bottom {k} entropy heads: {bottom_k_heads}")
# print(f"Top {k} entropy heads: {top_k_heads}")

In [ ]:
# Display attention heatmaps for top 4 and bottom 4 entropy heads
assert "avg_ent_l_h" in globals() and "sample0" in globals(), "Run entropy and attention collection cells first."

attn = sample0.detach().float().cpu().numpy()  # [T, L, H, C]
T, L, H, C = attn.shape

k = 4  # Number of heads to display (2x2 grid)

# Get top k (highest entropy) and bottom k (lowest entropy) heads
entropy_flat = avg_ent_l_h.reshape(-1)
indices_flat = np.argsort(entropy_flat)
top_k_indices = indices_flat[-k:]
bottom_k_indices = indices_flat[:k]

# Convert flat indices to (layer, head) tuples
bottom_k_heads = [(idx // H, idx % H) for idx in bottom_k_indices]
top_k_heads = [(idx // H, idx % H) for idx in top_k_indices[::-1]]

# Create figure with 2x2 grids side by side
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(13, 7))
gs = gridspec.GridSpec(2, 4, figure=fig, wspace=0.3, hspace=0.3, left=0.05, right=0.98, top=0.88, bottom=0.08)

# Add titles for each grid
fig.text(0.25, 0.92, "Lowest Entropy Heads", ha="center", fontsize=14, fontweight="bold")
fig.text(0.75, 0.92, "Highest Entropy Heads", ha="center", fontsize=14, fontweight="bold")

# Plot bottom k entropy heads (left 2x2 grid)
for i, (l, h) in enumerate(bottom_k_heads):
    row, col = i // 2, i % 2
    ax = fig.add_subplot(gs[row, col])
    im = ax.imshow(attn[:, l, h, :].T, aspect="equal", origin="lower", extent=(0, T, 0, C), cmap="bone_r")
    ax.set_xlabel("Prediction Timestep", fontsize=9)
    ax.set_xticks(np.arange(0, T + 128, 128))
    ax.set_ylabel("Context Timestep", fontsize=9)
    ax.set_yticks(np.arange(0, C, 128))
    ax.set_title(f"Layer {l}, Head {h}\n(Entropy: {avg_ent_l_h[l, h]:.2f})", fontsize=10)
    ax.tick_params(labelsize=8)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.ax.tick_params(labelsize=7)

# Plot top k entropy heads (right 2x2 grid)
for i, (l, h) in enumerate(top_k_heads):
    row, col = i // 2, i % 2 + 2  # Offset by 2 columns
    ax = fig.add_subplot(gs[row, col])
    attnT = attn[:, l, h, :].T
    vmax = np.max(attnT)
    print(f"vmax: {vmax}")
    effective_vmax = 0.1 if vmax > 0.5 else vmax
    im = ax.imshow(attnT, aspect="equal", origin="lower", extent=(0, T, 0, C), cmap="bone_r", vmax=effective_vmax)
    ax.set_xlabel("Prediction Timestep", fontsize=9)
    ax.set_xticks(np.arange(0, T + 128, 128))
    ax.set_ylabel("Context Timestep", fontsize=9)
    ax.set_yticks(np.arange(0, C, 128))
    ax.set_title(f"Layer {l}, Head {h}\n(Entropy: {avg_ent_l_h[l, h]:.2f})", fontsize=10)
    ax.tick_params(labelsize=8)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.ax.tick_params(labelsize=7)

plt.savefig(
    os.path.join(plot_save_dir, f"attention_heatmap_top_bottom_{k}_entropy_heads.pdf"), bbox_inches="tight", dpi=200
)
plt.show()

print(f"Bottom {k} entropy heads: {bottom_k_heads}")
print(f"Top {k} entropy heads: {top_k_heads}")

In [ ]:
# # Scatter grids: negative average entropy per head vs induction score for all config keys
# import pickle
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# # Paths
# model_name = "chronos-t5-base"
# base_dir = f"/work/results/rrt_induction_scores/{model_name}"

# # Ensure entropy has been computed
# assert "avg_ent_l_h" in globals(), "Run the entropy cell to compute avg_ent_l_h first."
# neg_avg_entropy = -np.asarray(avg_ent_l_h)  # shape [L, H]
# L, H = neg_avg_entropy.shape
# labels = [f"L{l}H{h}" for l in range(L) for h in range(H)]

# # List all config keys (directories)
# config_keys = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
# assert len(config_keys) > 0, f"No config directories found in {base_dir}"

# # Build subplot grid
# n = len(config_keys)
# cols = int(np.ceil(np.sqrt(n)))
# rows = int(np.ceil(n / cols))
# fig = make_subplots(rows=rows, cols=cols, shared_xaxes=True, shared_yaxes=True, subplot_titles=tuple(config_keys))

# def _get_mean(obj):
#     return np.asarray(obj["mean"]) if isinstance(obj, dict) and "mean" in obj else np.asarray(obj)

# # Add one scatter per config
# for i, ck in enumerate(config_keys):
#     row, col = divmod(i, cols)[0] + 1, divmod(i, cols)[1] + 1

#     center_pkl = os.path.join(base_dir, ck, "center_scores.pkl")
#     right_pkl = os.path.join(base_dir, ck, "right_scores.pkl")
#     if not (os.path.exists(center_pkl) and os.path.exists(right_pkl)):
#         print(f"Skipping {ck}: missing pkl files")
#         continue

#     with open(center_pkl, "rb") as f:
#         center_scores_mean = _get_mean(pickle.load(f))
#     with open(right_pkl, "rb") as f:
#         right_scores_mean = _get_mean(pickle.load(f))

#     induction_mean = np.maximum(center_scores_mean, right_scores_mean)

#     if induction_mean.shape != neg_avg_entropy.shape:
#         print(f"Skipping {ck}: shape mismatch {induction_mean.shape} vs {neg_avg_entropy.shape}")
#         continue

#     x_vals = neg_avg_entropy.reshape(-1)
#     y_vals = induction_mean.reshape(-1)

#     mask = np.isfinite(x_vals) & np.isfinite(y_vals)
#     r = np.corrcoef(x_vals[mask], y_vals[mask])[0, 1] if mask.sum() > 1 else np.nan
#     subtitle = f"{ck}<br><sup>r = {r:.3f if not np.isnan(r) else 'N/A'}</sup>"

#     # Update subplot title with correlation
#     fig.layout.annotations[i].text = subtitle

#     fig.add_trace(
#         go.Scatter(
#             x=x_vals, y=y_vals, mode="markers",
#             marker=dict(size=6, opacity=0.75, line=dict(width=0)),
#             text=labels,
#             hovertemplate="<b>%{text}</b><br>-Avg entropy: %{x:.3f}<br>Induction: %{y:.3f}<extra></extra>",
#             showlegend=False,
#         ),
#         row=row, col=col,
#     )

# # Axis labels on the outer plots
# for c in range(1, cols + 1):
#     fig.update_xaxes(title_text="-Avg entropy (nats)", row=rows, col=c)
# for r in range(1, rows + 1):
#     fig.update_yaxes(title_text="Induction score (max of center/right)", row=r, col=1)

# fig.update_layout(
#     title="Head -Entropy vs Induction across configs",
#     width=420 * cols, height=340 * rows,
#     hovermode="closest", template="plotly_white",
#     margin=dict(t=60, l=60, r=20, b=60),
# )

# fig.show()
